In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from utils import all_ukb_participants, build_expansion_pack, load_fid

from delphi import DAYS_PER_YEAR

In [ ]:
# --- Load CHIP data (UKB field 30105) ---
chip_df = load_fid("30105")  # rows = eid, cols = array indices
chip_df = chip_df[chip_df.notna().any(axis=1)]  # keep participants with any value

print(chip_df)
# Flatten + unique
chip_numbers = pd.Series(chip_df.values.ravel()).dropna().unique()
chip_numbers = chip_numbers.astype(int)

# Convert to strings (recommended for tokenizers)
chip_tokens = [f"chip_{n}" for n in chip_numbers]

# Add tokens one by one
for tok in chip_tokens:
    if tok not in tokenizer:
        tokenizer[tok] = len(tokenizer)

# Build lookup: number -> token id
lookup = {n: tokenizer[f"chip_{n}"] for n in chip_numbers}

print(lookup)

In [ ]:
print(len(chip_df))

In [ ]:
# # --- Filter to valid UKB participants ---

ukb_subjects = all_ukb_participants()
print("Top 20 UKB subjects in UKB:", ukb_subjects[:20])
print("Total UKB subjects in UKB:", len(ukb_subjects))

chip_subjects = chip_df.index.to_numpy().astype(int)
print("Top 20 CHIP subjects in UKB:", chip_subjects[:20])
print("Total CHIP subjects in UKB:", len(chip_subjects))

# Find CHIP subjects that are in UKB
is_valid = np.isin(chip_subjects, ukb_subjects)
valid_subjects = chip_subjects[is_valid]

# Make sure the type matches chip_df.index exactly
valid_subjects = valid_subjects.astype(str)

print("Top 20 valid CHIP subjects in UKB:", valid_subjects[:20])
print("Total valid CHIP subjects in UKB:", len(valid_subjects))

print(chip_df.index[:20])
# Filter chip_df safely
chip_df.index = chip_df.index.astype(str)
valid_subjects = valid_subjects.astype(str)
chip_df = chip_df.loc[chip_df.index.intersection(valid_subjects)]
print("Top 20 CHIP subjects in UKB:", chip_df.index[:20])
print("Total valid CHIP subjects in UKB:", len(chip_df))


In [ ]:
# --- Build token/time arrays ---
subjects = []
token_list = []
time_list = []
count_list = []

ukb_subjects = np.array([str(s) for s in ukb_subjects])  # all strings
chip_df.index = chip_df.index.astype(str)                  # all strings


for pid, row in chip_df.iterrows():
    # Drop missing numbers
    numbers = [int(v) for v in row.values if pd.notna(v)]
    if len(numbers) == 0:
        continue

    tokens = [lookup[v] for v in numbers]  # no .strip()
    timesteps = [0.0] * len(tokens)  # static features → time=0

    subjects.append(pid)
    token_list.extend(tokens)
    time_list.extend(timesteps)
    count_list.append(len(tokens))


In [ ]:
# Convert to numpy arrays
token_np = np.array(token_list, dtype=np.uint32)
time_np = np.array(time_list, dtype=np.float32)
count_np = np.array(count_list, dtype=np.uint32)
subjects = np.array(subjects, dtype=np.uint32)


In [ ]:
print(len(chip_df.index))
print(len(ukb_subjects))
print(token_np)
print(len(token_np))
print("Total subjects:", len(subjects_np))
print("Total tokens:", len(token_np))
print("Sum of counts:", count_np.sum())

In [ ]:
# --- Build expansion pack ---
build_expansion_pack(
    token_np=token_np,
    time_np=time_np,
    count_np=count_np,
    subjects=subjects,
    tokenizer=tokenizer,
    expansion_pack="chip",
    odir = "Delphi"
)